<Strong><span style="color:yellow;font-size:60px"> Analysis of news surrounding the Russia/Ukraine war</strong>
<span style="font-size:15px;color"><br> - import text from online sources
<br> - pre proccessing: cleaing, tokeisation, stop word removal, stemming/lemmatisation, vetorisation
<br> - Topic modelling: identification of pro russia vs pro-ukrainian sentiment
<br> - article summarisation (BERT)
<br> - model evaluation

In [1]:
import datetime
print(f"program started{datetime.datetime.now()}")
import pandas as pd
import requests

print(f"imported libraries{datetime.datetime.now()}")

program started2025-03-07 16:35:45.615220
imported libraries2025-03-07 16:35:45.925765


<span style="color:green;font-size:30px"> API hit and tabularisation of data

In [2]:
print(f"last api hit {datetime.datetime.now()}")

url = ('https://newsapi.org/v2/everything?'
       'q=ukraine&'
       'from=2025-02-07&'
       'sortBy=date&'
       'apiKey=8710d1e91328470981d652fb8496bfe0')

response = requests.get(url)

#break down response to tabulised data
data = response.json()
data = data['articles']

df = pd.DataFrame(data)
print(df.dtypes)


last api hit 2025-03-07 16:35:45.929305
source         object
author         object
title          object
description    object
url            object
urlToImage     object
publishedAt    object
content        object
dtype: object


<strong><span style="color:orange;font-size:50px">Text Processing</strong>
<br><span style="color:green"> - removal of special charachters and tags
<br> - tokenisation data data
<br> - stop word removal
<br> - vectorisation
<br> - lemminisation/stemming  </span>


In [ ]:
print(f"text processing started {datetime.datetime.now()}")

print(df.dtypes)
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from nltk.stem import WordNetLemmatizer
from nltk import ne_chunk, pos_tag
from textblob import TextBlob
#from nltk

nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('averaged_perceptron_tagger')

# Ensure NLTK data path is set
nltk.data.path.append('/Users/zacwells/nltk_data')

# Function to tokenize text with error handling
def safe_tokenize(text):
    try:
        return word_tokenize(text, preserve_line=True)
    except Exception as e:
        print(f"Tokenization error: {e}")
        return text.split()  # fallback to simple splitting

# Function to remove special characters
def removal_of_special_characters(text):
    pattern = r'[!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r]'
    clean_text = re.sub(pattern, '', text)
    return clean_text

def lemmatise_text(tokens):
    lemmatiser = WordNetLemmatizer()
    return [lemmatiser.lemmatize(token) for token in tokens]

def extract_entities(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    entities = ne_chunk(tagged)
    return entities

def get_sentiment(text):
    return TextBlob(text).sentiment.polarity


# Process the text columns
print("Starting text processing...")

# Convert to lowercase and remove special characters
for column in ['content', 'description', 'title']:
    df[column] = df[column].astype(str)
    df[column] = df[column].apply(removal_of_special_characters)
    
# Apply tokenization
print("Applying tokenization...")
df['content'] = df['content'].apply(safe_tokenize)
df['description'] = df['description'].apply(safe_tokenize)
df['title'] = df['title'].apply(safe_tokenize)

# Apply lemmatisation

print(f"lemmatisation started {datetime.datetime.now()}")
df['content'] = df['content'].apply(lemmatise_text)
df['description'] = df['description'].apply(lemmatise_text)
df['title'] = df['title'].apply(lemmatise_text)
print(f"lemmatisation complete {datetime.datetime.now()}")
print(f"shape of df {df.shape}")

# Convert lists of tokens back to strings
df['content_str'] = df['content'].apply(lambda x: ' '.join(x))
df['description_str'] = df['description'].apply(lambda x: ' '.join(x))
df['title_str'] = df['title'].apply(lambda x: ' '.join(x))

#apply NER to content
print(f"NER started {datetime.datetime.now()}")
df['content_entities'] = df['content_str'].apply(extract_entities)
print(f"NER complete {datetime.datetime.now()}")

print(f"sentiment analysis started {datetime.datetime.now()}")
df['content_sentiment'] = df['content_str'].apply(get_sentiment)
df['description_sentiment'] = df['description_str'].apply(get_sentiment)
print(f"sentiment analysis complete {datetime.datetime.now()}")

vector_small = TfidfVectorizer(min_df=2,
                        max_df=0.9,
                        stop_words='english',
                        ngram_range=(1,3))


print(f"content vectorization started {datetime.datetime.now()}")
context_vector_small = vector_small.fit_transform(df['content_str'])
print(f"content vectorization complete {datetime.datetime.now()}")

print(f"title vectorization started {datetime.datetime.now()}")
title_vector_small = vector_small.transform(df['title_str'])
print(f"title vectorization complete {datetime.datetime.now()}")

print(f"descripton vectorization started {datetime.datetime.now()}")
description_vector_small = vector_small.transform(df['description_str'])
print(f"descripton vectorization complete {datetime.datetime.now()}")

feature_names = vector_small.get_feature_names_out()

content_df = pd.DataFrame(
    context_vector_small.toarray(),
    columns=feature_names,
    index=df.index
)

title_df = pd.DataFrame(
    title_vector_small.toarray(),
    columns=feature_names,
    index=df.index
)

description_df = pd.DataFrame(
    description_vector_small.toarray(),
    columns=feature_names,
    index=df.index
)

content_df.columns = [f'content_{col}' for col in content_df.columns]
title_df.columns = [f'title_{col}' for col in title_df.columns]
description_df.columns = [f'description_{col}' for col in description_df.columns]

df_vectorised = pd.concat([df, content_df, title_df, description_df], axis=1)

print(f"shape of df_vectorised {df_vectorised.shape}")
print(f"number of features assed: {len(feature_names)}")

print("Text processing complete")

data cleaning started 2025-03-07 16:35:46.490652
source         object
author         object
title          object
description    object
url            object
urlToImage     object
publishedAt    object
content        object
dtype: object


[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/zacwells/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to /Users/zacwells/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/zacwells/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Starting text processing...
Applying tokenization...
lemmatisation started 2025-03-07 16:35:47.263204
lemmatisation complete 2025-03-07 16:35:48.032306
shape of df (98, 8)
NER started 2025-03-07 16:35:48.032971
NER complete 2025-03-07 16:35:57.702580
content vectorization started 2025-03-07 16:35:57.702687
content vectorization complete 2025-03-07 16:35:57.710782
title vectorization started 2025-03-07 16:35:57.710810
title vectorization complete 2025-03-07 16:35:57.711503
descripton vectorization started 2025-03-07 16:35:57.711517
descripton vectorization complete 2025-03-07 16:35:57.712429
shape of df_vectorised (98, 1266)
number of features assed: 418
Text processing complete


<strong><span style="font-size:50px;color:orange"> Topic Modeling Analysis</strong>
<br><span style="font-size:15px"> - key word ID for talking points
<br> - Theme analysis
<br> - political persuasion for Pro/Amti russian sentiment

In [ ]:
pro_russian_terms = ['special miliarty operation','denazification','russian protection',
                     'donetsk people republic','demiliterisation','nato encroachment','liberation of russian people',
                     'glory to russia','kyiv regime','Banderovtsy','russophobia','new russia','Novorossiya']
pro_ukrainian_terms = []
pro_western_terms = []
pro_putin_terms = []
pro_zelensky_terms = []
pro_nato_terms = []
pro_eu_terms = []
pro_trump_terms = []
pro_vance_terms = []